# Fleet Partnership Prediction - Competition Notebook

## Competition Overview

This notebook provides a **baseline solution** for the Fleet Partnership Prediction Challenge.

**Goal**: Predict whether a prospective company will become a partner (1) or not (0) based on 100 features.

**Evaluation Metric**: AUC-ROC (Area Under the Receiver Operating Characteristic Curve)

**Target**: Highest AUC to support your company's decision!

---

This notebook demonstrates a ML pipeline including:
- Data loading and exploration
- Missing value handling
- Outlier detection and removal
- Feature normalization
- Multiple modeling approaches
- Cross-validation for model selection
- Submission file generation

**Course**: Part of كورس البداية www.aibdaya.com AI & Data Science course by Dr. Mahmoud Eid  
**Website**: www.aibdaya.com

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Configuration

In [ ]:
# Deep Neural Network Configuration
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.001


TRAIN_PATH = '/kaggle/input/fleet-partnership-prediction/train.csv'
TEST_PATH = '/kaggle/input/fleet-partnership-prediction/test.csv'

print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Training Data: {TRAIN_PATH}")
print(f"  Testing Data: {TEST_PATH}")

Configuration:
  Epochs: 100
  Batch Size: 64
  Learning Rate: 0.001
  Training Data: /kaggle/input/fleet-partnership-prediction/train.csv
  Testing Data: /kaggle/input/fleet-partnership-prediction/test.csv


## 3. Load Data

Load the training and testing datasets:
- Training data: 100 features + target variable
- Testing data: 100 features only (no target - this is what your model should predict)

In [ ]:
def load_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    """
    Load training and testing data from CSV files.

    Returns:
        X_train, y_train, X_test: Feature matrices and training target
    """
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Separate features and target for training data
    X_train = train_df.drop('target', axis=1).values
    y_train = train_df['target'].values

    # Test data - drop 'id' column if present, and 'target' if present
    cols_to_drop = [col for col in ['id', 'target'] if col in test_df.columns]
    if cols_to_drop:
        X_test = test_df.drop(cols_to_drop, axis=1).values
    else:
        X_test = test_df.values

    print(f"Training data: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"Test data: {X_test.shape[0]} samples, {X_test.shape[1]} features")
    print(f"Missing values in training: {np.sum(np.isnan(X_train))}")
    print(f"Class distribution in training: {np.bincount(y_train)}")

    return X_train, y_train, X_test

# Load the data
X_train, y_train, X_test = load_data()


LOADING DATA
Training data: 38400 samples, 100 features
Test data: 9600 samples, 100 features
Missing values in training: 38399
Class distribution in training: [20263 18137]


## 4. Handle Missing Values

We use **median imputation** to handle missing values:
- More robust to outliers than mean imputation
- Calculate imputation values from training data only
- Apply the same values to test data (no data leakage)

In [ ]:
def handle_missing_values(X_train, X_test, strategy='median'):
    """
    Impute missing values using training statistics.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix
        strategy: 'median' or 'mean'

    Returns:
        X_train_imputed, X_test_imputed: Imputed feature matrices
    """
    print("\n" + "="*50)
    print("HANDLING MISSING VALUES")
    print("="*50)

    X_train_imputed = X_train.copy()
    X_test_imputed = X_test.copy()

    if strategy == 'median':
        impute_values = np.nanmedian(X_train, axis=0)
        print(f"Using median imputation (robust to outliers)")
    else:
        impute_values = np.nanmean(X_train, axis=0)
        print(f"Using mean imputation")

    for feature_idx in range(X_train.shape[1]):
        missing_mask_train = np.isnan(X_train_imputed[:, feature_idx])
        X_train_imputed[missing_mask_train, feature_idx] = impute_values[feature_idx]

        # Test set (using training statistics)
        missing_mask_test = np.isnan(X_test_imputed[:, feature_idx])
        X_test_imputed[missing_mask_test, feature_idx] = impute_values[feature_idx]

    print(f"Missing values after imputation:")
    print(f"  Training: {np.sum(np.isnan(X_train_imputed))}")
    print(f"  Test: {np.sum(np.isnan(X_test_imputed))}")

    return X_train_imputed, X_test_imputed

X_train, X_test = handle_missing_values(X_train, X_test, strategy='median')


HANDLING MISSING VALUES
Using median imputation (robust to outliers)
Missing values after imputation:
  Training: 0
  Test: 0


## 5. Detect and Remove Outliers

We use the **IQR (Interquartile Range) method** to detect outliers:
- Calculate Q1 (25th percentile) and Q3 (75th percentile)
- IQR = Q3 - Q1
- Outliers are values < Q1 - 1.5*IQR or > Q3 + 1.5*IQR


In [ ]:
import numpy as np

def cap_outliers_percentile(X_train, X_test, lower_percentile=5, upper_percentile=95):
    """
    Caps outliers in both Train and Test sets using statistics derived
    ONLY from the Training set to prevent data leakage.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix
        lower_percentile: Lower bound percentile (default 5)
        upper_percentile: Upper bound percentile (default 95)

    Returns:
        X_train_capped, X_test_capped: Transformed feature matrices
    """
    print("\n" + "="*50)
    print(f"CAPPING OUTLIERS (Winsorization)")
    print(f"Bounds derived from Train: {lower_percentile}th - {upper_percentile}th percentiles")
    print("="*50)

    X_train_capped = X_train.copy()
    X_test_capped = X_test.copy()
    n_features = X_train.shape[1]

    total_capped_train = 0
    total_capped_test = 0

    for feature_idx in range(n_features):
        train_feature_data = X_train[:, feature_idx]

        lower_bound = np.percentile(train_feature_data, lower_percentile)
        upper_bound = np.percentile(train_feature_data, upper_percentile)

        outliers_train = (X_train_capped[:, feature_idx] < lower_bound) | \
                         (X_train_capped[:, feature_idx] > upper_bound)
        total_capped_train += np.sum(outliers_train)

        X_train_capped[:, feature_idx] = np.clip(
            X_train_capped[:, feature_idx], lower_bound, upper_bound
        )

        outliers_test = (X_test_capped[:, feature_idx] < lower_bound) | \
                        (X_test_capped[:, feature_idx] > upper_bound)
        total_capped_test += np.sum(outliers_test)

        X_test_capped[:, feature_idx] = np.clip(
            X_test_capped[:, feature_idx], lower_bound, upper_bound
        )


    return X_train_capped, X_test_capped

# Apply capping/winsorization
X_train, X_test = cap_outliers_percentile(X_train, X_test, lower_percentile=5, upper_percentile=95)

print(f"Shape of training: {X_train.shape}")
print(f"Shape of testing: {X_test.shape}")


CAPPING OUTLIERS (Winsorization)
Bounds derived from Train: 5th - 95th percentiles
Shape of training: (38400, 100)
Shape of testing: (9600, 100)


## 6. Check Feature Correlations

Check for highly correlated features.

In [ ]:
def check_correlation(X, threshold=0.8):
    """
    Check for highly correlated features.

    Args:
        X: Feature matrix
        threshold: Correlation threshold

    Returns:
        corr_matrix: Correlation matrix
    """
    print("\n" + "="*50)
    print("CHECKING FEATURE CORRELATIONS")
    print("="*50)

    # Calculate correlation matrix
    corr_matrix = np.corrcoef(X.T)

    # Find high correlations (excluding diagonal)
    high_corr_pairs = []
    n_features = X.shape[1]

    for i in range(n_features):
        for j in range(i+1, n_features):
            if abs(corr_matrix[i, j]) > threshold:
                high_corr_pairs.append((i, j, corr_matrix[i, j]))

    if len(high_corr_pairs) == 0:
        print(f"No high correlations found (threshold={threshold})")
        print(f"No significant multicollinearity detected")
    else:
        print(f"Found {len(high_corr_pairs)} feature pairs with correlation > {threshold}")
        for i, j, corr in high_corr_pairs[:5]:  # Show first 5
            print(f"  - Feature {i} & Feature {j}: {corr:.3f}")

    return corr_matrix

# Check correlations
corr_matrix = check_correlation(X_train, threshold=0.8)


CHECKING FEATURE CORRELATIONS
No high correlations found (threshold=0.8)
No significant multicollinearity detected


## 7. Feature Normalization (Z-Score)

Apply **Z-score normalization** to standardize features
Calculate statistics from training data only to avoid data leakage

In [ ]:
def apply_zscore(X_train, X_test):
    """
    Apply Z-score normalization.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix

    Returns:
        X_train_scaled, X_test_scaled: Normalized feature matrices
    """
    print("\n" + "="*50)
    print("APPLYING Z-SCORE NORMALIZATION")
    print("="*50)

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    # Avoid division by zero
    std[std == 0] = 1

    # Apply to both train and test
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    print(f"Normalized {X_train.shape[1]} features")
    print(f"Training data: mean={np.mean(X_train_scaled):.6f}, std={np.std(X_train_scaled):.6f}")
    print(f"Test data: mean={np.mean(X_test_scaled):.6f}, std={np.std(X_test_scaled):.6f}")

    return X_train_scaled, X_test_scaled

X_train_scaled, X_test_scaled = apply_zscore(X_train, X_test)


APPLYING Z-SCORE NORMALIZATION
Normalized 100 features
Training data: mean=-0.000000, std=1.000000
Test data: mean=-0.000302, std=1.001269


## 8. Model Training and Evaluation Functions

Define helper functions for model training and evaluation using cross-validation.

In [ ]:
def evaluate_model_cv(model, X, y, model_type='sklearn'):
    """
    Evaluate model on training data using AUC-ROC.

    Args:
        model: Trained model
        X: Feature matrix
        y: Target vector
        model_type: 'sklearn' or 'pytorch'

    Returns:
        auc_score: AUC-ROC score on training data
    """
    if model_type == 'sklearn':
        y_pred_proba = model.predict_proba(X)[:, 1]
    else:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X)
            y_pred_proba = model(X_tensor).numpy().flatten()

    auc_score = roc_auc_score(y, y_pred_proba)
    return auc_score

## 9. Experiment 1: Logistic Regression (Baseline)


In [ ]:
def train_logistic_regression(X_train, y_train, penalty='l2', C=1.0):
    """
    Train Logistic Regression model.

    Args:
        X_train: Training features
        y_train: Training target
        penalty: Regularization type ('l1' or 'l2')
        C: Inverse regularization strength

    Returns:
        model: Trained model
    """
    print(f"\nTraining Logistic Regression (penalty={penalty}, C={C})")
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver='liblinear',
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)

    if penalty == 'l1':
        n_nonzero = np.sum(model.coef_ != 0)
        print(f"L1 regularization selected {n_nonzero} non-zero features")

    print(f"Model trained successfully")
    return model

def cross_validate_model(X_train, y_train, penalty='l2', C=1.0, n_folds=5):
    """
    Perform k-fold cross-validation for Logistic Regression.
    """
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver='liblinear',
        max_iter=1000,
        random_state=42
    )

    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=n_folds,
        scoring='roc_auc',
        n_jobs=-1
    )

    return np.mean(cv_scores), np.std(cv_scores), cv_scores

print("\n" + "="*50)
print("EXPERIMENT 1: Logistic Regression (Baseline)")
print("="*50)

# Cross-validation to estimate performance
lr_cv_mean, lr_cv_std, lr_cv_scores = cross_validate_model(X_train_scaled, y_train, penalty='l2', C=1.0, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"Individual folds: {lr_cv_scores}")

# Train final model on all training data
lr_model = train_logistic_regression(X_train_scaled, y_train, penalty='l2', C=1.0)
lr_train_auc = evaluate_model_cv(lr_model, X_train_scaled, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"Logistic Regression Results:")
print(f"  Training AUC-ROC: {lr_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 1: Logistic Regression (Baseline)

5-Fold CV AUC-ROC: 0.5107 +/- 0.0054
Individual folds: [0.5169529  0.5019617  0.50710292 0.51352062 0.51395509]

Training Logistic Regression (penalty=l2, C=1.0)
Model trained successfully

Logistic Regression Results:
  Training AUC-ROC: 0.5343
  CV AUC-ROC: 0.5107 +/- 0.0054


## 10. Experiment 2: PCA + Logistic Regression


In [ ]:
def apply_pca(X_train, X_test, n_components=30):
    """
    Apply PCA dimensionality reduction.
    """
    print(f"\nApplying PCA (n_components={n_components})")

    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    explained_variance = np.sum(pca.explained_variance_ratio_)

    print(f"Reduced from {X_train.shape[1]} to {n_components} principal components")
    print(f"Explained variance: {explained_variance*100:.2f}%")

    return X_train_pca, X_test_pca, pca

print("\n" + "="*50)
print("EXPERIMENT 2: PCA + Logistic Regression")
print("="*50)

X_train_pca, X_test_pca, pca = apply_pca(X_train_scaled, X_test_scaled, n_components=30)

# Cross-validation
lr_pca_cv_mean, lr_pca_cv_std, _ = cross_validate_model(X_train_pca, y_train, penalty='l2', C=1.0, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_pca_cv_mean:.4f} +/- {lr_pca_cv_std:.4f}")

# Train final model
lr_pca_model = train_logistic_regression(X_train_pca, y_train, penalty='l2', C=1.0)
lr_pca_train_auc = evaluate_model_cv(lr_pca_model, X_train_pca, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"PCA + Logistic Regression Results:")
print(f"  Training AUC-ROC: {lr_pca_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_pca_cv_mean:.4f} +/- {lr_pca_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 2: PCA + Logistic Regression

Applying PCA (n_components=30)
Reduced from 100 to 30 principal components
Explained variance: 30.83%

5-Fold CV AUC-ROC: 0.5023 +/- 0.0064

Training Logistic Regression (penalty=l2, C=1.0)
Model trained successfully

PCA + Logistic Regression Results:
  Training AUC-ROC: 0.5167
  CV AUC-ROC: 0.5023 +/- 0.0064


## 11. Experiment 3: L1 Regularization (Sparse Feature Selection)

Use L1 regularization for embedded feature selection.

In [ ]:
print("\n" + "="*50)
print("EXPERIMENT 3: L1 Regularization + Logistic Regression")
print("="*50)

# Cross-validation
lr_l1_cv_mean, lr_l1_cv_std, _ = cross_validate_model(X_train_scaled, y_train, penalty='l1', C=0.1, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_l1_cv_mean:.4f} +/- {lr_l1_cv_std:.4f}")

# Train final model
lr_l1_model = train_logistic_regression(X_train_scaled, y_train, penalty='l1', C=0.1)
lr_l1_train_auc = evaluate_model_cv(lr_l1_model, X_train_scaled, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"L1 Regularization Results:")
print(f"  Training AUC-ROC: {lr_l1_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_l1_cv_mean:.4f} +/- {lr_l1_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 3: L1 Regularization + Logistic Regression

5-Fold CV AUC-ROC: 0.5113 +/- 0.0053

Training Logistic Regression (penalty=l1, C=0.1)
L1 regularization selected 87 non-zero features
Model trained successfully

L1 Regularization Results:
  Training AUC-ROC: 0.5342
  CV AUC-ROC: 0.5113 +/- 0.0053


Chance level is 50% (0 or 1) and this's the worst classification, thus **we ensured that the data is *NON-LINEAR* and in this case we'll go with DNN approach**

## 12. Experiment 4: Deep Neural Network

Train a deep neural network to capture non-linear patterns:

**Architecture:**
- Input Layer: 100 features
- Hidden Layer 1: 512 neurons + ReLU + Dropout(0.2)
- Hidden Layer 2: 256 neurons + ReLU + Dropout(0.2)
- Hidden Layer 3: 128 neurons + ReLU + Dropout(0.2)
- Hidden Layer 4: 64 neurons + ReLU + Dropout(0.2)
- Hidden Layer 5: 32 neurons + ReLU + Dropout(0.2)
- Output Layer: 1 neuron + Sigmoid

In [ ]:
class DeepNN(nn.Module):
    """
    Deep Neural Network for binary classification.
    """
    def __init__(self, input_dim=100):
        super(DeepNN, self).__init__()
        self.layer_1 = nn.Linear(input_dim, 512)
        self.layer_2 = nn.Linear(512, 256)
        self.layer_3 = nn.Linear(256, 128)
        self.layer_4 = nn.Linear(128, 64)
        self.layer_5 = nn.Linear(64, 32)
        self.layer_out = nn.Linear(32, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, inputs):
        x = self.relu(self.layer_1(inputs))
        x = self.dropout(x)
        x = self.relu(self.layer_2(x))
        x = self.dropout(x)
        x = self.relu(self.layer_3(x))
        x = self.dropout(x)
        x = self.relu(self.layer_4(x))
        x = self.dropout(x)
        x = self.relu(self.layer_5(x))
        x = self.dropout(x)
        x = self.sigmoid(self.layer_out(x))
        return x

def train_dnn_epoch(model, loader, optimizer, criterion):
    """Train the DNN for one epoch."""
    model.train()
    for _, (X_batch, y_batch) in enumerate(loader):
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def train_pytorch_model(model, X_train, y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """
    Train PyTorch model.
    """
    # Convert to PyTorch tensors
    X_tensor = torch.FloatTensor(X_train)
    y_tensor = torch.FloatTensor(y_train).reshape(-1, 1)

    # Create DataLoader
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    print(f"Training for {epochs} epochs...")
    for epoch in range(epochs):
        train_dnn_epoch(model, dataloader, optimizer, criterion)
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs} completed")

    return model

In [ ]:
def cross_validate_dnn(X_train, y_train, learning_rates, n_folds=5):
    """
    Perform k-fold cross-validation for DNN with different learning rates.
    """
    print(f"\n{n_folds}-Fold Cross-Validation for Learning Rate")
    print(f"Testing learning rates: {learning_rates}")

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    results = {lr: [] for lr in learning_rates}

    for lr in learning_rates:
        print(f"\n  Testing LR = {lr}...")

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            model = DeepNN(input_dim=X_train.shape[1])
            model = train_pytorch_model(model, X_tr, y_tr, learning_rate=lr, epochs=EPOCHS, batch_size=BATCH_SIZE)

            auc = evaluate_model_cv(model, X_val, y_val, model_type='pytorch')
            results[lr].append(auc)
            print(f"    Fold {fold+1}: AUC = {auc:.4f}")

        avg_auc = np.mean(results[lr])
        std_auc = np.std(results[lr])
        print(f"  LR={lr}: Mean AUC-ROC = {avg_auc:.4f} +/- {std_auc:.4f}")

    best_lr = max(learning_rates, key=lambda lr: np.mean(results[lr]))
    best_auc = np.mean(results[best_lr])

    print(f"\nBest learning rate: {best_lr} (Mean AUC-ROC = {best_auc:.4f})")

    return best_lr, results

print("\n" + "="*50)
print("EXPERIMENT 4: Deep Neural Network")
print("="*50)

print(f"\nArchitecture: Input(100) -> 512 -> 256 -> 128 -> 64 -> 32 -> Output(1)")
print(f"Activation: ReLU")
print(f"Regularization: Dropout(0.2)")

# Cross-validation to find best learning rate
learning_rates_to_test = [0.01, 0.001]
best_lr, cv_results_deep = cross_validate_dnn(
    X_train_scaled, y_train, learning_rates_to_test, n_folds=5
)

# Train final model with best learning rate on all training data
print(f"\nTraining final model with LR={best_lr}...")
deep_nn = DeepNN(input_dim=X_train_scaled.shape[1])
deep_nn = train_pytorch_model(deep_nn, X_train_scaled, y_train, learning_rate=best_lr, epochs=EPOCHS)

dnn_train_auc = evaluate_model_cv(deep_nn, X_train_scaled, y_train, model_type='pytorch')
dnn_cv_mean = np.mean(cv_results_deep[best_lr])
dnn_cv_std = np.std(cv_results_deep[best_lr])

print(f"\n{'='*50}")
print(f"Deep Neural Network Results:")
print(f"  CV AUC-ROC: {dnn_cv_mean:.4f} +/- {dnn_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 4: Deep Neural Network

Architecture: Input(100) -> 512 -> 256 -> 128 -> 64 -> 32 -> Output(1)
Activation: ReLU
Regularization: Dropout(0.2)

5-Fold Cross-Validation for Learning Rate
Testing learning rates: [0.01, 0.001]

  Testing LR = 0.01...
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 1: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 2: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 3: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 4: AUC = 0.5000
Training for 100 epochs...
  E

## 13. Final Results Summary

Compare models based on cross-validation performance.

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

results_df = pd.DataFrame({
    'Model': [
        'Logistic Regression (L2)',
        'PCA + Logistic Regression',
        'L1 Regularization',
        'Deep Neural Network'
    ],
    'CV AUC': [
        lr_cv_mean,
        lr_pca_cv_mean,
        lr_l1_cv_mean,
        dnn_cv_mean
    ],
    'CV Std': [
        lr_cv_std,
        lr_pca_cv_std,
        lr_l1_cv_std,
        dnn_cv_std
    ]
})

print("\n", results_df.to_string(index=False))
print("\n" + "="*60)
print(f"Best Model (by CV): {results_df.loc[results_df['CV AUC'].idxmax(), 'Model']}")
print(f"Best CV AUC: {results_df['CV AUC'].max():.4f}")
print("="*60)



FINAL RESULTS SUMMARY

                     Model   CV AUC   CV Std
 Logistic Regression (L2) 0.510699 0.005421
PCA + Logistic Regression 0.502334 0.006438
        L1 Regularization 0.511300 0.005287
      Deep Neural Network 0.762150 0.008863

Best Model (by CV): Deep Neural Network
Best CV AUC: 0.7621


## 14. Generate Submission File

Create a submission CSV with predictions from your best model.



## **IMPORTANT**: This is the file you submit to Kaggle .. so follow the same formatting!

In [ ]:
def generate_submission(model, X_test, model_type='pytorch', filename='submission.csv'):
    """
    Generate submission file for Kaggle competition.

    Args:
        model: Trained model
        X_test: Test features (scaled)
        model_type: 'sklearn' or 'pytorch'
        filename: Output filename

    Returns:
        submission_df: DataFrame with predictions
    """
    print("\n" + "="*50)
    print("GENERATING SUBMISSION FILE")
    print("="*50)

    if model_type == 'sklearn':
        predictions = model.predict_proba(X_test)[:, 1]
    else:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_test)
            predictions = model(X_tensor).numpy().flatten()

    # IMPORTANT: Kaggle format: id, target
    submission_df = pd.DataFrame({
        'id': range(len(predictions)),
        'target': predictions
    })

    submission_df.to_csv(filename, index=False)

    print(f"\nSubmission file saved: {filename}")
    print(f"Total predictions: {len(predictions)}")
    print(f"\nPrediction statistics:")
    print(f"  Min: {predictions.min():.4f}")
    print(f"  Max: {predictions.max():.4f}")
    print(f"  Mean: {predictions.mean():.4f}")
    print(f"\nFirst 10 predictions:")
    print(submission_df.head(10).to_string(index=False))

    return submission_df

# Generate submission with Deep NN (best baseline model)
print("\nUsing Deep Neural Network for submission...")
submission = generate_submission(deep_nn, X_test_scaled, model_type='pytorch', filename='submission.csv')

print("\n" + "="*50)
print("SUBMISSION READY!")
print("="*50)
print("\nUpload 'submission.csv' to Kaggle to see your score!")
print("Good luck reaching >0.98 AUC-ROC!")


Using Deep Neural Network for submission...

GENERATING SUBMISSION FILE

Submission file saved: submission.csv
Total predictions: 9600

Prediction statistics:
  Min: 0.0000
  Max: 1.0000
  Mean: 0.4533

First 10 predictions:
 id   target
  0 0.506188
  1 0.017038
  2 0.951456
  3 0.119161
  4 0.913562
  5 0.000218
  6 0.996934
  7 0.488424
  8 0.803452
  9 0.026325

SUBMISSION READY!

Upload 'submission.csv' to Kaggle to see your score!
Good luck reaching >0.98 AUC-ROC!
